# Stepwise Regression Analysis - VPCA + Spectral Library

**Author:** Abdulrahman Hussein  
**Supervisor:** Dr. Joseph D. Ortiz  
**Institution:** Kent State University, Department of Earth Sciences  
**Version:** 1.2.0

---

This is a thin orchestrator notebook. All heavy logic lives in the `framework/` Python package.

**Workflow (6 cells):**

1. **Imports & Config** - paths and thresholds
2. **Load Data** - VPCA loadings + spectral library (auto-aligned)
3. **Run Analysis** - bidirectional stepwise per VPC
4. **Export** - multi-sheet Excel + CSV summary
5. **Visualize** - per-component and overview plots

**Algorithm (bidirectional only):**

1. Forward step: add the most significant candidate (p < `p_enter`) that doesn't exceed VIF
2. Backward step (p-value): remove any variable with p > `p_remove`
3. Backward step (VIF): remove any variable with VIF > `vif_threshold`
4. Stop on convergence or `max_iter`

**Excel output sheets:** Summary, VIF_Details, IterationHistory, Descriptive_Stats, Settings, plus one per-VPC sheet with SPSS-style Model Summary / ANOVA / Coefficients tables.

## Cell 1 - Imports & Configuration

In [ ]:
# Cell 2: Imports & Config
import sys, os
from pathlib import Path

# Add repo root to sys.path so 'framework' is importable from any cwd
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "framework").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / "framework").exists():
    sys.path.insert(0, str(REPO_ROOT.parent))
    REPO_ROOT = REPO_ROOT.parent

from framework import VPCAStepwiseAnalysis
from framework import (
    plot_iteration_history, plot_vif, plot_coefficients,
    plot_component, plot_all_components,
)
import matplotlib.pyplot as plt

# --- Configuration ---
LOADINGS_FILE  = REPO_ROOT / "learning" / "test-data" / "test_loadings.csv"
LIBRARY_FILE   = REPO_ROOT / "learning" / "test-data" / "test_library.csv"
OUTPUT_DIR     = REPO_ROOT / "output"

P_ENTER        = 0.05    # threshold to ADD a variable
P_REMOVE       = 0.10    # threshold to REMOVE a variable
VIF_THRESHOLD  = 2.0     # max acceptable VIF
MAX_ITER       = 100
VERBOSE        = 1       # 0=silent, 1=step-by-step, 2=detailed

print(f"Repo root : {REPO_ROOT}")
print(f"Loadings  : {LOADINGS_FILE}")
print(f"Library   : {LIBRARY_FILE}")
print(f"Output    : {OUTPUT_DIR}")

## Cell 3 - Load Data

Auto-detects wavelength columns, aligns wavelengths between loadings and library, and runs a z-score sanity check (data is assumed pre-z-scored).

In [ ]:
# Cell 4: Load Data
analyzer = VPCAStepwiseAnalysis(
    p_enter=P_ENTER,
    p_remove=P_REMOVE,
    vif_threshold=VIF_THRESHOLD,
    max_iter=MAX_ITER,
    verbose=VERBOSE,
)
z_loadings, z_library = analyzer.load_data(str(LOADINGS_FILE), str(LIBRARY_FILE))

print(f"\nLoadings shape : {z_loadings.shape}")
print(f"Library shape  : {z_library.shape}")
z_loadings.head()

## Cell 5 - Run Analysis

Runs bidirectional stepwise selection for **each VPC component**, building all SPSS-style tables in memory.

In [ ]:
# Cell 6: Run Analysis
summary_df, detailed_df = analyzer.run_analysis()

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("DETAILED (first 15 rows)")
print("=" * 70)
print(detailed_df.head(15).to_string(index=False))

## Cell 7 - Export Results

Writes a multi-sheet Excel file plus a CSV summary.

In [ ]:
# Cell 8: Export
files = analyzer.export_results(str(OUTPUT_DIR))
for k, v in files.items():
    print(f"{k:15s} -> {v}")

## Cell 9 - Visualize

All plots use the helpers from `framework.plots`. Pick any component name (e.g. `VPCA_Z1`).

In [ ]:
# Cell 10: Plots
# Overview across all components
plot_all_components(analyzer)
plt.show()

# Pick a component to inspect
component = analyzer.summary_results_.iloc[0]["Component"]
print(f"Plotting details for {component}")

plot_component(analyzer, component)
plt.show()

model = analyzer.get_model(component)
plot_iteration_history(model)
plt.show()
plot_vif(model)
plt.show()
plot_coefficients(model)
plt.show()

print(model.summary())